[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/expressive_rendering/03_rule_based_rendering.ipynb)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

HOME_DIR = "."

if IN_COLAB:
    import os
    import sys

    # Install system dependency (FluidSynth executable)
    ! apt-get update -qq
    ! apt-get install -y -qq fluidsynth

    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop pyfluidsynth appdirs

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone --recurse-submodules https://github.com/MusicalInformatics/miws26.git
    # RuMEx is a git submodule of the repository (this also gets it if the repository was already cloned)
    !git -C miws26 submodule update --init --recursive
    sys.path.insert(0, "/content/miws26/expressive_rendering/")
    sys.path.insert(0, "/content/miws26/expressive_rendering/rumex/")
    HOME_DIR = "/content/miws26/expressive_rendering/"

# RuMEx: rules for expressive performance

In the following notebooks, a *model* will learn how to perform a score from data. Before that, we try the other way around: we write down *rules* (e.g., "play notes marked *f* loudly", "make the melody stand out") and apply them to a score. This is the idea behind rule-based systems such as the KTH rule system (Friberg et al., 2006). We use **RuMEx**, a small Python package built on [partitura](https://github.com/CPJKU/partitura), which covers four expressive dimensions: dynamics, tempo, timing, and articulation.

In this notebook we will:

1. see how the rules are implemented and how to add new ones,
2. write a rule that plays the main melody louder: we find the melody with a skyline algorithm that gives a strictly monophonic melody, and use a sigmoid to keep the loudness below the maximum velocity,
3. see how the rule changes the MIDI velocity of the notes in a piano roll, and
4. render a performance with a set of rules and listen to it using FluidSynth.

In [ ]:
import os
import sys
import inspect
import warnings

import numpy as np
import matplotlib.pyplot as plt
import partitura as pt
from IPython.display import HTML, Audio, display
from partitura.utils.fluidsynth import HAS_FLUIDSYNTH, synthesize_fluidsynth

# RuMEx lives in the folder `rumex` (a package with the same name inside it). It is a git submodule:
# if the folder is empty, run `git submodule update --init --recursive` in the repository
sys.path.insert(0, os.path.join(HOME_DIR, "rumex"))
import rumex
from rumex import (
    load_score,
    render_piece,
    compute_note_features,
    expanded_note_array,
    double_notes_keep_mask,
    DEFAULT_CONFIG,
    DEFAULT_DYNAMICS_PIPELINE,
)
from rumex.dynamics import monophonic_skyline

from helper.rumex_plots import animate_velocity_pianoroll

warnings.filterwarnings("ignore", module="partitura")

We use an excerpt of Mozart's variations K. 265 (one of the example scores in `sample_data`), which has a melody over an accompaniment.

In [ ]:
score_fn = os.path.join(HOME_DIR, "sample_data", "musicxml", "Mozart_K265_1.musicxml")
part = load_score(score_fn)
print(f"{len(part.note_array())} notes")

## 1. Rules for expressive performance

### 1.1 How RuMEx renders a performance

RuMEx goes through the following steps (see `rumex.render_piece`):

1. **Note features.** For each note of the score we compute features such as the dynamics marking that is active (`p`, `mf`, `f`, ...), whether it has a staccato, or whether it is inside a crescendo. These are the [note features of partitura](https://partitura.readthedocs.io/en/latest/modules/partitura.musicanalysis.html) (similar to the basis functions of the previous notebook).
2. **Rules.** There are four *rule pipelines*, one for each expressive dimension: `DynamicsRules`, `TempoRules`, `TimingRules`, and `ArticulationRules`. A pipeline is a list of rules that are applied one after the other.
3. **Expressive parameters.** The result of the pipelines are the expressive parameters of each note: MIDI velocity, beat period (tempo), timing deviation, and articulation.
4. **Performance codec.** The parameters are turned into a performance (a `PerformedPart`) with the performance codec of partitura.

**Grace notes.** In the score, grace notes have no duration. Before applying the rules, `render_piece` gives them a position and a duration in the score that depends on the type of ornament (this is done by `expand_grace_notes_from_local_grace_order` in partitura): an *acciaccatura* starts before the beat and takes time from the previous note, an *appoggiatura* starts on the beat and takes time from the main note. From then on they are treated as any other note.

**Double notes.** Composers sometimes write the same pitch at the same time in two voices. A pianist cannot play it twice (and a mechanical piano like a Disklavier would have problems with it), so `render_piece` keeps only one of them (the longest one) before applying the rules.

In [ ]:
W, names = compute_note_features(part)
print(W.shape)
print(names[:20])

The default pipelines are lists of the names of the rules:

In [ ]:
for dimension, cfg in DEFAULT_CONFIG.items():
    print(f"{dimension}:")
    for rule in cfg["pipeline"]:
        print("   ", rule)

### 1.2 The API for adding rules

A rule is just a **function** that gets information about the score and the expressive parameters computed so far by the previous rules in the pipeline, and returns the updated parameters. The dynamics rules, for example, receive and return `velocity_trend` and `velocity_dev`, both in MIDI velocity units (0-127), one value per note. The final velocity of a note is `velocity_trend - velocity_dev`. The idea is that `velocity_trend` is the loudness given by the markings of the score, and `velocity_dev` the deviations from the trend (hence it is *subtracted*: a positive deviation makes the note softer, a negative one louder).

This is one of the rules that RuMEx already has, the simplest one (random fluctuations of the loudness, which imitate the motor noise of a human performer):

In [ ]:
print(inspect.getsource(rumex.dynamics.motor_noise_rules))

Every rule of the dynamics pipeline has this form:

```python
def my_rule(velocity_trend, velocity_dev, <parameters of the rule>=<default>, *args, **kwargs):
    ...
    return velocity_trend, velocity_dev
```

The pipeline calls each rule with the following *keyword arguments*; the rule picks the ones it needs by name and the rest goes into `**kwargs` (this is why every rule has `*args, **kwargs`):

| argument | description |
| --- | --- |
| `W`, `names` | note features: array (notes x features) and the names of its columns |
| `pitch` | MIDI pitch of each note |
| `onsets`, `duration` | score onset and duration of each note (in beats) |
| `unique_onsets`, `unique_onset_idxs` | the distinct onsets, and the indices of the notes that start at each of them |
| `velocity_trend`, `velocity_dev` | expressive parameters computed so far (in MIDI velocity) |
| `rng` | random number generator, use it if the rule has a random component (so that results are reproducible) |

All the arrays have one entry per note, in the order of `part.note_array()`. The other pipelines work in the same way, but with their own parameters (tempo rules, for example, return the beat period of each note).

An entry of a pipeline can be

* the **name** of a rule that is defined in RuMEx (a string, as in the lists above),
* a **function** that you wrote yourself, or
* a tuple `(rule, {"parameter": value})`, to set the parameters of a rule.

As an example, here is a toy rule that makes high notes louder, with a parameter `slope` (MIDI velocity per semitone):

In [ ]:
def pitch_slope_rules(pitch, velocity_trend, velocity_dev, slope=0.0, *args, **kwargs):
    # remember: velocity = velocity_trend - velocity_dev, so to make notes louder we subtract
    velocity_dev -= slope * (pitch - pitch.mean())
    return velocity_trend, velocity_dev

To use it, we make a configuration where the dynamics pipeline has the default rules plus our new one, and we call `render_piece`. We fix the seed of the random number generator, so that the only difference between the performances is the new rule:

In [ ]:
def render(dynamics_pipeline, seed=0, config=DEFAULT_CONFIG):
    """Render a performance of `part` with a given dynamics pipeline (the other pipelines are the default ones)"""
    config = {k: dict(v) for k, v in config.items()}
    config["dynamics"]["pipeline"] = dynamics_pipeline
    return render_piece(part, config=config, rng=np.random.RandomState(seed))


# `render_piece` removes the double notes (same pitch and onset in two voices): the ones that are kept
keep = double_notes_keep_mask(part)
note_ids = part.note_array()["id"][keep]


def velocities(ppart):
    """MIDI velocity of each note, in the order of the notes of the score"""
    by_id = {n["id"]: n["velocity"] for n in ppart.notes}
    return np.array([by_id[i] for i in note_ids])


pitch = part.note_array()["pitch"][keep]
for slope in [0, 0.5, 1]:
    pipeline = DEFAULT_DYNAMICS_PIPELINE + [(pitch_slope_rules, {"slope": slope})]
    v = velocities(render(pipeline))
    print(f"slope {slope}: correlation between pitch and velocity {np.corrcoef(pitch, v)[0, 1]:.2f}")

Numbers are not enough: let us listen to it. We synthesize the performances with FluidSynth (through partitura), first without the rule (`slope = 0`) and then with `slope = 1`, where the high notes get louder and the low notes softer:

In [ ]:
SAMPLE_RATE = 44100
SECONDS = 40


def listen(ppart, seconds=SECONDS):
    assert HAS_FLUIDSYNTH, "FluidSynth is not available (see the installation instructions of partitura)"
    audio = synthesize_fluidsynth(note_info=ppart, samplerate=SAMPLE_RATE)
    return Audio(audio[: seconds * SAMPLE_RATE], rate=SAMPLE_RATE)


for slope in [0, 1]:
    print(f"slope = {slope}")
    pipeline = DEFAULT_DYNAMICS_PIPELINE + [(pitch_slope_rules, {"slope": slope})]
    display(listen(render(pipeline)))

### 1.3 Implementing the melody loudness rule

Pianists often play the main melody louder than the accompaniment, so that the listener can follow it. To write a rule for that, we first need to know which notes are the melody. We use the skyline algorithm from last week, in the version of RuMEx that gives a strictly monophonic melody, i.e., the selected notes never overlap (`monophonic_skyline`). It returns a boolean array with the melody notes:

In [ ]:
note_array = expanded_note_array(part)[keep]  # the notes as the rules see them (grace notes expanded, no double notes)
is_melody = monophonic_skyline(note_array["pitch"], note_array["onset_beat"], note_array["duration_beat"])
print(f"{is_melody.sum()} melody notes out of {len(is_melody)}")

Now we write the rule. It has a parameter `melody_relative_loudness`:

* `0`: the melody has the same loudness as the rest (the rule does nothing),
* a positive value makes the melody louder by that proportion (`0.5`: 50% louder; `1`: twice as loud),
* a negative value makes it softer (`-0.5`: 50% softer).

Remember that `velocity = velocity_trend - velocity_dev`. The rule finds the melody notes and changes `velocity_dev` for them. Here is a first version:

In [ ]:
def melody_loudness_linear_rules(
    pitch, onsets, duration, velocity_trend, velocity_dev, melody_relative_loudness=0.0, *args, **kwargs
):
    is_melody = monophonic_skyline(pitch, onsets, duration)
    # the velocity of the melody notes is multiplied by (1 + melody_relative_loudness)
    velocity = velocity_trend[is_melody] - velocity_dev[is_melody]
    new_velocity = velocity * (1 + melody_relative_loudness)
    velocity_dev[is_melody] = velocity_trend[is_melody] - new_velocity
    return velocity_trend, velocity_dev

We add the rule at the **end** of the dynamics pipeline (so that it acts on the velocity that the other rules computed), with the parameter in a tuple, and compare the velocity of the melody and the accompaniment for a few values. The velocity of a performed note is always clipped to the range 1-127 when the performance is created, so notes cannot be louder than 127.

In [ ]:
for r in [-0.5, 0, 0.5, 1]:
    pipeline = DEFAULT_DYNAMICS_PIPELINE + [(melody_loudness_linear_rules, {"melody_relative_loudness": r})]
    v = velocities(render(pipeline))
    print(f"melody_relative_loudness {r:+.1f}: mean velocity melody {v[is_melody].mean():5.1f}, "
          f"accompaniment {v[~is_melody].mean():5.1f}")

#### A smooth upper limit

Clipping has a problem: all the notes that would be louder than 127 get exactly the same velocity, so the differences between them (the phrasing of the melody) are lost. Instead of clipping, we can bend the increase with a **sigmoid**, so that the velocity approaches 127 smoothly without going above it (after rounding to an integer, it can still end at 127 for very large increases).

Let `v` be the current velocity of a melody note and `delta = melody_relative_loudness * v` the increase that the linear rule would apply. The room that is left until 127 is `headroom = 127 - v`. We replace the increase by

$$\Delta = \text{headroom} \cdot \big(2\,\sigma(2\,\text{delta} / \text{headroom}) - 1\big), \qquad \sigma(x) = \frac{1}{1 + e^{-x}}$$

For small increases (compared to the headroom) $\Delta \approx$ `delta`, as in the linear rule, but $\Delta$ is never larger than `headroom`. Decreases cannot go below zero (for `melody_relative_loudness >= -1`), so we keep them linear.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def melody_loudness_rules(
    pitch,
    onsets,
    duration,
    velocity_trend,
    velocity_dev,
    melody_relative_loudness=0.0,
    max_velocity=127,
    *args,
    **kwargs,
):
    is_melody = monophonic_skyline(pitch, onsets, duration)
    velocity = velocity_trend[is_melody] - velocity_dev[is_melody]
    delta = melody_relative_loudness * velocity
    if melody_relative_loudness > 0:
        headroom = np.maximum(max_velocity - velocity, 1e-6)
        delta = headroom * (2 * sigmoid(2 * delta / headroom) - 1)
    velocity_dev[is_melody] = velocity_trend[is_melody] - (velocity + delta)
    return velocity_trend, velocity_dev

Let us compare both versions for notes of different velocities (this is the new velocity of a melody note as a function of `melody_relative_loudness`):

In [ ]:
r = np.linspace(0, 3, 100)
fig, ax = plt.subplots(figsize=(6, 3.5))
for v0, color in zip([40, 70, 100], ["C0", "C1", "C2"]):
    linear = np.minimum(v0 * (1 + r), 127)
    delta = r * v0
    headroom = 127 - v0
    soft = v0 + headroom * (2 * sigmoid(2 * delta / headroom) - 1)
    ax.plot(r, linear, "--", color=color)
    ax.plot(r, soft, color=color, label=f"velocity {v0}")
ax.axhline(127, color="k", linewidth=0.5)
ax.set_xlabel("melody_relative_loudness")
ax.set_ylabel("new MIDI velocity")
ax.set_title("dashed: linear and clipped, solid: sigmoid")
ax.legend(frameon=False)
fig.tight_layout()

RuMEx has the linear version of this rule (`rumex.dynamics.melody_loudness_rules`). The one we just wrote, with the sigmoid, is the one we use in the rest of the notebook:

In [ ]:
for r in [-0.5, 0, 0.5, 1, 2]:
    pipeline = DEFAULT_DYNAMICS_PIPELINE + [(melody_loudness_rules, {"melody_relative_loudness": r})]
    v = velocities(render(pipeline))
    print(f"melody_relative_loudness {r:+.1f}: mean velocity melody {v[is_melody].mean():5.1f}, "
          f"accompaniment {v[~is_melody].mean():5.1f}, maximum {v.max()}")

## 2. Seeing the effect on the velocity

To see what the rules do, we show the piano roll of the performance (computed with `partitura.utils.compute_pianoroll`) for different values of `melody_relative_loudness`. The color of each note is its MIDI velocity, the melody notes (according to the monophonic skyline) have a white dot, and the notes with velocity 127 (the maximum) are red. We show the first 25 seconds. First the linear rule, where the velocity is clipped at 127:

In [ ]:
def render_melody(r, rule=melody_loudness_rules):
    pipeline = DEFAULT_DYNAMICS_PIPELINE + [(rule, {"melody_relative_loudness": r})]
    return render(pipeline)


anim = animate_velocity_pianoroll(
    lambda r: render_melody(r, melody_loudness_linear_rules),
    values=np.linspace(-1, 2, 13),
    melody_ids=note_array["id"][is_melody],
    seconds=25,
)
HTML(anim.to_jshtml())

With negative values the melody becomes softer than the accompaniment, until it nearly disappears at `-1`. With positive values, the melody gets louder, and at some point the loudest notes reach 127 and cannot get any louder (red). Now the same with the sigmoid rule:

In [ ]:
anim = animate_velocity_pianoroll(
    render_melody,
    values=np.linspace(-1, 2, 13),
    melody_ids=note_array["id"][is_melody],
    seconds=25,
)
HTML(anim.to_jshtml())

Now the loudest notes get close to 127 instead of being cut at 127, and the louder notes of the melody are still louder than the softer ones.

## 3. Rendering a performance

Finally, we render a performance with a full set of rules. We use the default rules for tempo, timing, and articulation (see section 1.1), and for the dynamics the default rules plus our melody loudness rule. The pipelines of the dimensions can be changed in the same way as the dynamics pipeline; the dynamics rules, for example, are:

* `dynamics_markings_rules`: velocity of the markings (*p*, *mf*, *f*, ...) and crescendo/diminuendo,
* `simple_accent_rules`: accents, staccato, and tenuto,
* `polynomial_pitch_height_rules`: very high and very low notes are softer,
* `motor_noise_rules`: random fluctuations.

In [ ]:
config = {k: dict(v) for k, v in DEFAULT_CONFIG.items()}
config["dynamics"]["pipeline"] = DEFAULT_DYNAMICS_PIPELINE + [
    (melody_loudness_rules, {"melody_relative_loudness": 0.5})
]

ppart = render_piece(part, config=config, rng=np.random.RandomState(0))
print(f"{len(ppart.notes)} notes, {ppart.notes[-1]['note_off']:.0f} seconds")

A performance can be saved as a MIDI file with partitura:

In [ ]:
pt.save_performance_midi(ppart, "my_rumex_performance.mid")

We listen to it (up to 40 seconds). For comparison, we also synthesize the same performance without the melody rule (`melody_relative_loudness = 0`):

In [ ]:
print("Without the melody rule:")
display(listen(render_melody(0)))
print("With melody_relative_loudness = 0.5:")
display(listen(ppart))

## 4. Performing another piece

The rules can be applied to any score. As an example, we render *Tuileries*, from Mussorgsky's *Pictures at an Exhibition* (the score is in the folder `generation_task`), with the same configuration as above:

In [ ]:
mussorgsky_fn = os.path.join(
    HOME_DIR, "generation_task", "mussorgsky-pictures-at-an-exhibition-06-no-3-tuileries.musicxml"
)
mussorgsky = load_score(mussorgsky_fn)
mussorgsky_ppart = render_piece(mussorgsky, config=config, rng=np.random.RandomState(0))
print(f"{len(mussorgsky_ppart.notes)} notes, {mussorgsky_ppart.notes[-1]['note_off']:.0f} seconds")
pt.save_performance_midi(mussorgsky_ppart, "my_rumex_mussorgsky.mid")
display(listen(mussorgsky_ppart, seconds=90))

## References

* A. Friberg, R. Bresin, and J. Sundberg (2006). Overview of the KTH rule system for musical performance. *Advances in Cognitive Psychology*, 2(2-3), 145-161.
* A. L. Uitdenbogerd and J. Zobel (1999). Melodic matching techniques for large music databases. In *Proceedings of the 7th ACM International Conference on Multimedia*.